In [ ]:
import json
import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import numpy as np

plt.style.use("seaborn-v0_8")

: 

In [ ]:

# Locate all water_calibration.json files
root_dir = "./data"   

json_files = []
for path, dirs, files in os.walk(root_dir):
    for f in files:
        if f == "water_calibration.json":
            json_files.append(os.path.join(path, f))

print(f"Found {len(json_files)} calibration files")
json_files


In [ ]:

def load_and_recompute(json_path):
    with open(json_path, "r") as f:
        data = json.load(f)

    measurements = data["input"]["measurements"]

    # Extract valve_open_time and water_weight
    times = []
    weights = []

    for m in measurements:
        times.append(m["valve_open_time"])
        # water_weight is a list, but always length 1
        weights.append(m["water_weight"][0])

    X = np.array(times).reshape(-1, 1)
    y = np.array(weights)

    # Fit linear regression
    model = LinearRegression()
    model.fit(X, y)

    slope = model.coef_[0]
    intercept = model.intercept_
    r2 = model.score(X, y)

    df = pd.DataFrame({"valve_open_time": times, "water_weight": weights})

    return df, slope, intercept, r2


In [ ]:

# Process all calibration files

results = []

for jf in json_files:
    df, slope, intercept, r2 = load_and_recompute(jf)
    results.append({
        "file": jf,
        "slope": slope,
        "intercept": intercept,
        "r2": r2,
        "df": df
    })

results


In [ ]:
# Plot each calibration curve
for r in results:
    df = r["df"]
    slope = r["slope"]
    intercept = r["intercept"]

    x = df["valve_open_time"]
    y_pred = slope * x + intercept

    plt.figure(figsize=(6,4))
    plt.scatter(df["valve_open_time"], df["water_weight"], label="Measured")
    plt.plot(x, y_pred, color="red", label=f"Fit: y={slope:.4f}x + {intercept:.4f}")
    plt.title(f"Calibration: {r['file']}")
    plt.xlabel("Valve Open Time (s)")
    plt.ylabel("Water Weight (g)")
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:

# Summary plot of slopes across directories

plt.figure(figsize=(8,4))

slopes = [r["slope"] for r in results]
labels = [os.path.basename(os.path.dirname(r["file"])) for r in results]

plt.bar(labels, slopes)
plt.ylabel("Slope (g per second)")
plt.title("Recomputed Slopes Across Calibration Files")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
